In [ ]:
import pandas as pd

# 1. 加载整个 2.xlsx 文件
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')

# 2. 打印真实表名
print("2.xlsx 里的真实表名：", xls.sheet_names)

# 3. 按位置读取（绝不出错）
df_load = pd.read_excel(xls, sheet_name=1, header=0)  # 第 1 张表
df_pv = pd.read_excel(xls, sheet_name=0, header=0)    # 第 0 张表

# 4. 读取 1.xlsx
df_price = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)

# 5. 打印列名和行数
print("负载列名：", df_load.columns.tolist())
print("光伏列名：", df_pv.columns.tolist())
print("负载行数：", len(df_load))
print("光伏行数：", len(df_pv))

# 6. 直接提取数据
prices = df_price['电价'].values.astype(float)
loads = df_load.iloc[:, -1].values.astype(float) # 取最后一列数据
pvs = df_pv.iloc[:, -1].values.astype(float)    # 取最后一列数据

print(f"时间点数量: {len(prices)}")

In [6]:
import pandas as pd
import pulp
import numpy as np

# 1. 读取数据
df = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df['电价'].values.astype(float)
loads = df['小区负载'].values.astype(float)
pvs_actual = df['光伏发电预测功率'].values.astype(float)  # 真实光伏

T = 144
dt = 1/6

# 2. 模拟不同时刻的预测误差
np.random.seed(42)
forecast_0 = np.clip(pvs_actual + np.random.normal(0, 0.30, T), 0, None)  # 0:00 预测（误差30%）

# 
prob_plan = pulp.LpProblem("Microgrid_Q3_Plan", pulp.LpMinimize)
plan_buy = pulp.LpVariable.dicts("plan_buy", range(T), lowBound=0)
plan_charge = pulp.LpVariable.dicts("plan_charge", range(T), lowBound=0, upBound=5000/6)
plan_discharge = pulp.LpVariable.dicts("plan_discharge", range(T), lowBound=0, upBound=5000/6)
plan_soc = pulp.LpVariable.dicts("plan_soc", range(T+1), lowBound=1200, upBound=10800)

# 计划阶段的目标：基于有误差的预测，最小化计划购电成本
prob_plan += pulp.lpSum([prices[t] * plan_buy[t] for t in range(T)])
for t in range(T):
    prob_plan += plan_buy[t] + (forecast_0[t] * dt) + plan_discharge[t] * dt == loads[t] * dt + plan_charge[t] * dt
    prob_plan += plan_soc[t+1] == plan_soc[t] + 0.9 * plan_charge[t] - (1/0.9) * plan_discharge[t]
prob_plan += plan_soc[0] == 6000
prob_plan += plan_soc[T] >= 5900
prob_plan += plan_soc[T] <= 6100

prob_plan.solve()
print("0:00 计划求解状态:", pulp.LpStatus[prob_plan.status])

# 提取0:00的计划结果，作为6:00调整的基准
plan_buy_values = [plan_buy[t].varValue for t in range(T)]

# 3. 6:00 调整阶段：基于“真实的实际光伏”，修正计划
prob_adjust = pulp.LpProblem("Microgrid_Q3_Adjust", pulp.LpMinimize)

adjust_buy = pulp.LpVariable.dicts("adjust_buy", range(T), lowBound=0)
emergency = pulp.LpVariable.dicts("emergency", range(T), lowBound=0)
curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
adjust_charge = pulp.LpVariable.dicts("adjust_charge", range(T), lowBound=0, upBound=5000/6)
adjust_discharge = pulp.LpVariable.dicts("adjust_discharge", range(T), lowBound=0, upBound=5000/6)
adjust_soc = pulp.LpVariable.dicts("adjust_soc", range(T+1), lowBound=1200, upBound=10800)

over_buy = pulp.LpVariable.dicts("over_buy", range(T), lowBound=0)
under_buy = pulp.LpVariable.dicts("under_buy", range(T), lowBound=0)

# 调整阶段目标：买电费 + 紧急购电费 + 违约金
prob_adjust += pulp.lpSum([
    prices[t] * plan_buy_values[t] +  # 计划费用（固定值）
    5 * prices[t] * emergency[t] +
    0.5 * prices[t] * over_buy[t] +
    1.5 * prices[t] * under_buy[t]
    for t in range(T)
])

for t in range(T):
    prob_adjust += adjust_buy[t] + emergency[t] + (pvs_actual[t] * dt - curtail[t]) + adjust_discharge[t] * dt == loads[t] * dt + adjust_charge[t] * dt
    prob_adjust += adjust_soc[t+1] == adjust_soc[t] + 0.9 * adjust_charge[t] - (1/0.9) * adjust_discharge[t]
    # 违约金定义！调整量与计划量的差
    prob_adjust += adjust_buy[t] - plan_buy_values[t] == under_buy[t] - over_buy[t]
    prob_adjust += adjust_charge[t] <= 5000 / 6
    prob_adjust += adjust_discharge[t] <= 5000 / 6

prob_adjust += adjust_soc[0] == 6000
prob_adjust += adjust_soc[T] >= 5900
prob_adjust += adjust_soc[T] <= 6100

prob_adjust.solve()
print("\n6:00 调整求解状态:", pulp.LpStatus[prob_adjust.status])

if pulp.LpStatus[prob_adjust.status] == 'Optimal':
    print(f"最优总费用: {pulp.value(prob_adjust.objective):.2f} 元")
    for t in range(10):
        print(f"时间 {t+1}: 计划买电 {plan_buy_values[t]:.2f}, 调整买电 {adjust_buy[t].varValue:.2f}, 违约金(多/少) {over_buy[t].varValue:.2f}/{under_buy[t].varValue:.2f}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/53566016f60e45ebb8bd2bc7c181cc87-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/53566016f60e45ebb8bd2bc7c181cc87-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 296 COLUMNS
At line 1452 RHS
At line 1744 BOUNDS
At line 2323 ENDATA
Problem MODEL has 291 rows, 577 columns and 1011 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve determined that the problem was infeasible with tolerance of 1e-08
Analysis indicates model infeasible or unbounded
Perturbing problem by 0.001% of 0.44345807 - largest nonzero change 1.4219134e-07 ( 8.7557754e-05%) - largest zero change 1.4199726e-07
0  Obj 0.013980403 Primal inf 223557.73 (146)
80  Obj 12279.228 Primal inf 387443.19 (171)
160  Obj 31016.847 Primal inf 172805.32 (1

In [10]:
import pandas as pd
import pulp
import numpy as np

# 1. 读取原始数据
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')
df_load = pd.read_excel(xls, sheet_name=1, header=0)
df_pv = pd.read_excel(xls, sheet_name=0, header=0)
df_price = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df_price['电价'].values.astype(float)

df_load['日期'] = pd.to_datetime(df_load.iloc[:, 0]).dt.strftime('%Y-%m-%d')
df_pv['日期'] = pd.to_datetime(df_pv.iloc[:, 0]).dt.strftime('%Y-%m-%d')

all_dates = df_load['日期'].unique()
target_dates = [d for d in all_dates if '2025-02-01' <= d <= '2025-12-31']
print(f"共需处理 {len(target_dates)} 天的数据...")

T = 144
dt = 1/6
np.random.seed(42)

plan_records = []
adjust_records = []
cd_records = []
emergency_records = []

# 2. 循环每一天，跑模型
for i, date_str in enumerate(target_dates):
    
    if i % 10 == 0:
        print(f"正在处理第 {i+1}/{len(target_dates)} 天: {date_str}")

    load_row = df_load[df_load['日期'] == date_str]
    pv_row = df_pv[df_pv['日期'] == date_str]
    
    if load_row.empty or pv_row.empty:
        continue
        
    loads = pd.to_numeric(load_row.iloc[0, 1:145], errors='coerce').fillna(0).values.astype(float)
    pvs_actual = pd.to_numeric(pv_row.iloc[0, 1:145], errors='coerce').fillna(0).values.astype(float)
    
    # 模拟 0:00 的预测误差
    forecast_0 = np.clip(pvs_actual + np.random.normal(0, 0.30, T), 0, None)

    #  加了 try...except，防止某一天报错导致整个循环崩掉
    try:
        # --- 阶段一：0:00 制定计划 ---
        prob_plan = pulp.LpProblem(f"Plan_{date_str}", pulp.LpMinimize)
        plan_buy = pulp.LpVariable.dicts("plan_buy", range(T), lowBound=0)
        plan_charge = pulp.LpVariable.dicts("plan_charge", range(T), lowBound=0, upBound=5000/6)
        plan_discharge = pulp.LpVariable.dicts("plan_discharge", range(T), lowBound=0, upBound=5000/6)
        plan_soc = pulp.LpVariable.dicts("plan_soc", range(T+1), lowBound=1200, upBound=10800)

        prob_plan += pulp.lpSum([prices[t] * plan_buy[t] for t in range(T)])
        for t in range(T):
            prob_plan += plan_buy[t] + (forecast_0[t] * dt) + plan_discharge[t] * dt == loads[t] * dt + plan_charge[t] * dt
            prob_plan += plan_soc[t+1] == plan_soc[t] + 0.9 * plan_charge[t] - (1/0.9) * plan_discharge[t]
        prob_plan += plan_soc[0] == 6000
        prob_plan += plan_soc[T] >= 5900
        prob_plan += plan_soc[T] <= 6100
        prob_plan.solve()
        
        plan_buy_values = [plan_buy[t].varValue if plan_buy[t].varValue else 0 for t in range(T)]

        # --- 阶段二：6:00 调整 ---
        prob_adjust = pulp.LpProblem(f"Adjust_{date_str}", pulp.LpMinimize)
        adjust_buy = pulp.LpVariable.dicts("adjust_buy", range(T), lowBound=0)
        emergency = pulp.LpVariable.dicts("emergency", range(T), lowBound=0)
        curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
        adjust_charge = pulp.LpVariable.dicts("adjust_charge", range(T), lowBound=0, upBound=5000/6)
        adjust_discharge = pulp.LpVariable.dicts("adjust_discharge", range(T), lowBound=0, upBound=5000/6)
        adjust_soc = pulp.LpVariable.dicts("adjust_soc", range(T+1), lowBound=1200, upBound=10800)
        over_buy = pulp.LpVariable.dicts("over_buy", range(T), lowBound=0)
        under_buy = pulp.LpVariable.dicts("under_buy", range(T), lowBound=0)

        prob_adjust += pulp.lpSum([
            prices[t] * plan_buy_values[t] + 5 * prices[t] * emergency[t] +
            0.5 * prices[t] * over_buy[t] + 1.5 * prices[t] * under_buy[t]
            for t in range(T)
        ])
        for t in range(T):
            prob_adjust += adjust_buy[t] + emergency[t] + (pvs_actual[t] * dt - curtail[t]) + adjust_discharge[t] * dt == loads[t] * dt + adjust_charge[t] * dt
            prob_adjust += adjust_soc[t+1] == adjust_soc[t] + 0.9 * adjust_charge[t] - (1/0.9) * adjust_discharge[t]
            prob_adjust += adjust_buy[t] - plan_buy_values[t] == under_buy[t] - over_buy[t]
        prob_adjust += adjust_soc[0] == 6000
        prob_adjust += adjust_soc[T] >= 5900
        prob_adjust += adjust_soc[T] <= 6100
        prob_adjust.solve()

        if pulp.LpStatus[prob_adjust.status] != 'Optimal':
            continue

        # --- 记录结果 ---
        for t in range(T):
            time_str = f"{(t//6):02d}:{(t%6)*10:02d}:00"
            plan_records.append({'日期': date_str, '时间段': time_str, '计划购电量(kWh)': plan_buy_values[t]})
            adjust_records.append({'日期': date_str, '时间段': time_str, '调整购电量(kWh)': adjust_buy[t].varValue})
            cd_records.append({
                '日期': date_str, '时间段': time_str,
                '充电量(kWh)': adjust_charge[t].varValue,
                '放电量(kWh)': adjust_discharge[t].varValue,
                '0:00储电量(kWh)': adjust_soc[0].varValue if t == 0 else None,
                '24:00储电量(kWh)': adjust_soc[T].varValue if t == T - 1 else None
            })
            if emergency[t].varValue > 0.01:
                emergency_records.append({'日期': date_str, '时间段': time_str, '紧急购电量(kWh)': emergency[t].varValue})

    except Exception as e:
        # 如果某一天报错，打印出来，然后继续下一天
        print(f"❌ 第 {date_str} 天计算出错: {e}")
        continue

# 3. 保存最终结果
with pd.ExcelWriter('result3.xlsx') as writer:
    pd.DataFrame(plan_records).to_excel(writer, sheet_name='计划购电量', index=False)
    pd.DataFrame(adjust_records).to_excel(writer, sheet_name='调整购电量', index=False)
    pd.DataFrame(cd_records).to_excel(writer, sheet_name='充放电量', index=False)
    if len(emergency_records) > 0:
        pd.DataFrame(emergency_records).to_excel(writer, sheet_name='紧急购电量', index=False)
    else:
        pd.DataFrame(columns=['日期', '时间段', '紧急购电量(kWh)']).to_excel(writer, sheet_name='紧急购电量', index=False)

print(f"\n✅ result3.xlsx 已生成！总计跑了 {len(plan_records)//144} 天。")

共需处理 334 天的数据...
正在处理第 1/334 天: 2025-02-01
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/1694d00fb07b43babce1a76acd5d4245-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/1694d00fb07b43babce1a76acd5d4245-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 296 COLUMNS
At line 1452 RHS
At line 1744 BOUNDS
At line 2323 ENDATA
Problem MODEL has 291 rows, 577 columns and 1011 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve determined that the problem was infeasible with tolerance of 1e-08
Analysis indicates model infeasible or unbounded
Perturbing problem by 0.001% of 0.44345807 - largest nonzero change 1.4219134e-07 ( 8.7557754e-05%) - largest zero change 1.4199726e-07
0  Obj 0.013980403 Primal inf 183551.37 (146)
80  Obj 0.18449722 Primal inf 1456131.7 (183

In [13]:
import pandas as pd
import pulp
import numpy as np

# 1. 读取数据
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')
df_load = pd.read_excel(xls, sheet_name=1, header=0)
df_pv = pd.read_excel(xls, sheet_name=0, header=0)
df_price = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)
prices = df_price['电价'].values.astype(float)

df_load['日期'] = pd.to_datetime(df_load.iloc[:, 0]).dt.strftime('%Y-%m-%d')
df_pv['日期'] = pd.to_datetime(df_pv.iloc[:, 0]).dt.strftime('%Y-%m-%d')

all_dates = df_load['日期'].unique()
target_dates = [d for d in all_dates if '2025-04-10' <= d <= '2025-12-31']
print(f"共需处理 {len(target_dates)} 天的数据...")

T = 144
dt = 1/6
np.random.seed(42)

plan_records = []
adjust_records = []
cd_records = []
emergency_records = []

for i, date_str in enumerate(target_dates):
    if i % 10 == 0:
        print(f"正在处理第 {i+1}/{len(target_dates)} 天: {date_str}")

    load_row = df_load[df_load['日期'] == date_str]
    pv_row = df_pv[df_pv['日期'] == date_str]
    
    if load_row.empty or pv_row.empty:
        continue
        
    loads = pd.to_numeric(load_row.iloc[0, 1:145], errors='coerce').fillna(0).values.astype(float)
    pvs_actual = pd.to_numeric(pv_row.iloc[0, 1:145], errors='coerce').fillna(0).values.astype(float)
    
    # ⚠️ 核心修改：只要整天数据里有一个非零值，就保留这一天
    if np.sum(loads) == 0 and np.sum(pvs_actual) == 0:
        continue

    forecast_0 = np.clip(pvs_actual + np.random.normal(0, 0.30, T), 0, None)

    try:
        # --- 0:00 计划 ---
        prob_plan = pulp.LpProblem(f"Plan_{date_str}", pulp.LpMinimize)
        plan_buy = pulp.LpVariable.dicts("plan_buy", range(T), lowBound=0)
        plan_charge = pulp.LpVariable.dicts("plan_charge", range(T), lowBound=0, upBound=5000/6)
        plan_discharge = pulp.LpVariable.dicts("plan_discharge", range(T), lowBound=0, upBound=5000/6)
        plan_soc = pulp.LpVariable.dicts("plan_soc", range(T+1), lowBound=1200, upBound=10800)

        prob_plan += pulp.lpSum([prices[t] * plan_buy[t] for t in range(T)])
        for t in range(T):
            prob_plan += plan_buy[t] + (forecast_0[t] * dt) + plan_discharge[t] * dt == loads[t] * dt + plan_charge[t] * dt
            prob_plan += plan_soc[t+1] == plan_soc[t] + 0.9 * plan_charge[t] - (1/0.9) * plan_discharge[t]
        prob_plan += plan_soc[0] == 6000
        prob_plan += plan_soc[T] >= 5900
        prob_plan += plan_soc[T] <= 6100
        prob_plan.solve()
        plan_buy_values = [plan_buy[t].varValue if plan_buy[t].varValue else 0 for t in range(T)]

        # --- 6:00 调整 ---
        prob_adjust = pulp.LpProblem(f"Adjust_{date_str}", pulp.LpMinimize)
        adjust_buy = pulp.LpVariable.dicts("adjust_buy", range(T), lowBound=0)
        emergency = pulp.LpVariable.dicts("emergency", range(T), lowBound=0)
        curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
        adjust_charge = pulp.LpVariable.dicts("adjust_charge", range(T), lowBound=0, upBound=5000/6)
        adjust_discharge = pulp.LpVariable.dicts("adjust_discharge", range(T), lowBound=0, upBound=5000/6)
        adjust_soc = pulp.LpVariable.dicts("adjust_soc", range(T+1), lowBound=1200, upBound=10800)
        over_buy = pulp.LpVariable.dicts("over_buy", range(T), lowBound=0)
        under_buy = pulp.LpVariable.dicts("under_buy", range(T), lowBound=0)

        prob_adjust += pulp.lpSum([
            prices[t] * plan_buy_values[t] + 5 * prices[t] * emergency[t] +
            0.5 * prices[t] * over_buy[t] + 1.5 * prices[t] * under_buy[t]
            for t in range(T)
        ])
        for t in range(T):
            prob_adjust += adjust_buy[t] + emergency[t] + (pvs_actual[t] * dt - curtail[t]) + adjust_discharge[t] * dt == loads[t] * dt + adjust_charge[t] * dt
            prob_adjust += adjust_soc[t+1] == adjust_soc[t] + 0.9 * adjust_charge[t] - (1/0.9) * adjust_discharge[t]
            prob_adjust += adjust_buy[t] - plan_buy_values[t] == under_buy[t] - over_buy[t]
        prob_adjust += adjust_soc[0] == 6000
        prob_adjust += adjust_soc[T] >= 5900
        prob_adjust += adjust_soc[T] <= 6100
        prob_adjust.solve()

        if pulp.LpStatus[prob_adjust.status] != 'Optimal':
            continue

        for t in range(T):
            time_str = f"{(t//6):02d}:{(t%6)*10:02d}:00"
            plan_records.append({'日期': date_str, '时间段': time_str, '计划购电量(kWh)': plan_buy_values[t]})
            adjust_records.append({'日期': date_str, '时间段': time_str, '调整购电量(kWh)': adjust_buy[t].varValue})
            cd_records.append({
                '日期': date_str, '时间段': time_str,
                '充电量(kWh)': adjust_charge[t].varValue,
                '放电量(kWh)': adjust_discharge[t].varValue,
                '0:00储电量(kWh)': adjust_soc[0].varValue if t == 0 else None,
                '24:00储电量(kWh)': adjust_soc[T].varValue if t == T - 1 else None
            })
            if emergency[t].varValue > 0.01:
                emergency_records.append({'日期': date_str, '时间段': time_str, '紧急购电量(kWh)': emergency[t].varValue})

    except Exception as e:
        print(f"❌ 第 {date_str} 天计算出错: {e}")
        continue

# 3. 保存最终结果
with pd.ExcelWriter('result3.xlsx') as writer:
    pd.DataFrame(plan_records).to_excel(writer, sheet_name='计划购电量', index=False)
    pd.DataFrame(adjust_records).to_excel(writer, sheet_name='调整购电量', index=False)
    pd.DataFrame(cd_records).to_excel(writer, sheet_name='充放电量', index=False)
    if len(emergency_records) > 0:
        pd.DataFrame(emergency_records).to_excel(writer, sheet_name='紧急购电量', index=False)
    else:
        pd.DataFrame(columns=['日期', '时间段', '紧急购电量(kWh)']).to_excel(writer, sheet_name='紧急购电量', index=False)

print(f"\n✅ result3.xlsx 已生成！总计跑了 {len(plan_records)//144} 天。")

共需处理 266 天的数据...
正在处理第 1/266 天: 2025-04-10
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/74e7ef37e5d84db8be527e2026d7d873-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/74e7ef37e5d84db8be527e2026d7d873-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 296 COLUMNS
At line 1452 RHS
At line 1744 BOUNDS
At line 2323 ENDATA
Problem MODEL has 291 rows, 577 columns and 1011 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve determined that the problem was infeasible with tolerance of 1e-08
Analysis indicates model infeasible or unbounded
Perturbing problem by 0.001% of 0.44345807 - largest nonzero change 1.4219134e-07 ( 8.7557754e-05%) - largest zero change 1.4199726e-07
0  Obj 0.013980403 Primal inf 232239.96 (146)
65  Obj 0.17483392 Primal inf 1300652.2 (175